# PHYS 405 Computerized Data Analysis · Week 3 · Random variables and probability distributions
Instructor: Prof. Dr. Bora Işıldak · Department of Physics, Yeditepe University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/Phys_405_2026_2027_Fall/blob/main/Labs/Labs3/Lab3.ipynb)

**File → Save a copy in Drive, then work in your copy.** Your work lives in your Drive copy, not in the link.

How the lab works (eight problems, about 14 minutes each):
1. Read the problem. Write your prediction in the **Predict** cell.
2. Run the code, then **Compare** with your prediction.
3. Run the two checks. Each check tests the answer a different way.
4. Do the **Your turn** task in the empty cell.

Today's physics: six scintillator bars, a twelve-channel array, the dead zones of a 40 cm bar, the muon survival, a histogram of 2000 decay times, a voltmeter, a photomultiplier's dark counts, and a digital display. All simulated numbers use `np.random.default_rng(seed)` with the seed given in each problem, so your printed numbers match the slides.
The lab uses new data: the lecture's numbers do not carry over.

**Your Drive folder.** The next cell connects Google Drive (Colab asks for permission once) and makes the folder
`MyDrive/PHY405_Labs/Week03` if it does not exist. Every file this notebook writes goes there.

In [ ]:
import os
try:
    from google.colab import drive                      # running in Colab
    drive.mount("/content/drive")
    LAB_DIR = "/content/drive/MyDrive/PHY405_Labs/Week03"
except ImportError:                                     # running elsewhere: a local folder
    LAB_DIR = os.path.join(os.getcwd(), "PHY405_Labs", "Week03")
os.makedirs(LAB_DIR, exist_ok=True)
os.chdir(LAB_DIR)                                       # files written below land here
print("working folder:", os.getcwd())

In [ ]:
import numpy as np
from math import comb, erf, exp, sqrt
%config InlineBackend.figure_format = 'retina'

TAU = 2.1969811          # muon mean life in microseconds (PDG 2026 review)
print("numpy", np.__version__)

## Problem 1 · At least one bar fires
Six scintillator bars each fire independently with probability 0.2 on a passage. What is the probability that at least one bar fires? (Probabilities are illustrative.)

**Predict.** Adding the six probabilities gives $6 \times 0.2 = 1.2$; is that a probability? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
rng = np.random.default_rng(1)
hits = (rng.random((100_000, 6)) < 0.2).any(axis=1)
p1 = 1 - 0.8**6
print(round(p1, 3), round(hits.mean(), 3))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The printed numbers are the complement and the simulation.

In [ ]:
# check A (inequality): 0 <= P <= 1, and P >= 0.2 (one bar alone gives 0.2)
assert 0 <= p1 <= 1 and p1 >= 0.2
print("check A passed")

In [ ]:
# check B (simulation): the simulated fraction agrees with the formula to 0.01
assert abs(hits.mean() - p1) < 0.01
print("check B passed")

**Your turn.** Four bars each fire with probability 0.35, independently. What is the probability that at least one fires? Give the answer to three significant figures.

*Hint:* use the complement; a bar is silent with probability 0.65.

In [ ]:
# TODO: write your answer here

## Problem 2 · Exactly four of twelve
A twelve-channel array: each channel fires with probability 0.25 per passage, independently. What is the probability that exactly four channels fire on one passage?

**Predict.** The mean is $12 \times 0.25 = 3$; is exactly four likely, and about how likely? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
pmf = [comb(12, k) * 0.25**k * 0.75**(12 - k) for k in range(13)]
rng = np.random.default_rng(2)
sim2 = ((rng.random((100_000, 12)) < 0.25).sum(axis=1) == 4).mean()
print(round(pmf[4], 4), round(sum(pmf), 6))
print("P(3) =", round(pmf[3], 4), "  trap without the arrangement count:", round(0.25**4 * 0.75**8, 6))
print("simulation:", round(sim2, 4))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. Which is larger, $P(3)$ or $P(4)$? Why?

In [ ]:
# check A (simulation): 100 000 twelve-trial runs (seed 2) agree with the formula to 0.01
assert abs(sim2 - pmf[4]) < 0.01
print("check A passed")

In [ ]:
# check B (normalisation): the probabilities over k = 0..12 sum to 1
assert abs(sum(pmf) - 1) < 1e-9
print("check B passed")

**Your turn.** Eight trials each succeed with probability 0.5. What is the probability of at least six successes? Keep the answer as a fraction and as a decimal.

*Hint:* add the three terms for k = 6, 7 and 8; each arrangement has probability $0.5^8$.

In [ ]:
# TODO: write your answer here

## Problem 3 · Dead zones of a bar
A particle hits a scintillator bar 40 cm long; every position is equally likely (illustrative). The readout ignores hits within 3 cm of either end. What fraction of hits is lost? What is $P(10\ \text{cm} < x < 15\ \text{cm})$?

*Givens:* uniform density on $[0, L]$, $L = 40$ cm; a probability is an area: density × width; the bar has two ends.

**Predict.** Is the loss $3/40 = 7.5\ \%$? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
L, d = 40.0, 3.0                           # cm
x = np.random.default_rng(3).uniform(0, L, 100_000)
lost = ((x < d) | (x > L - d)).mean()
print(2 * d / L, round(lost, 4))
print("P(10 < x < 15 cm) =", round(5 / L, 3), "  trap, one end only:", round(d / L, 3))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(x, bins=np.arange(0, 40.5, 1), density=True, color="tab:blue", alpha=0.6, label="100 000 simulated hits")
ax.axhline(1 / L, color="k", lw=1.5, label="density 1/L = 0.025 per cm")
ax.axvspan(0, d, color="tab:red", alpha=0.25, label="dead zones")
ax.axvspan(L - d, L, color="tab:red", alpha=0.25)
ax.axvspan(10, 15, color="tab:green", alpha=0.25, label="10 < x < 15 cm")
ax.set_xlabel("hit position x (cm)"); ax.set_ylabel("density (1/cm)")
ax.set_title("Problem 3 · a flat density: every probability is a rectangle")
ax.legend(fontsize=8, loc="lower center"); plt.tight_layout(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The first number is $2d/L$; the second is the simulated loss. How many dead zones did your prediction count?

In [ ]:
# check A (normalisation and units): (1/L per cm) x (L cm) = 1, dimensionless
assert abs((1 / L) * L - 1) < 1e-12
print("check A passed")

In [ ]:
# check B (simulation): 100 000 hits (seed 3) lose 0.1516, within 0.01 of 2d/L = 0.150
assert abs(lost - 2 * d / L) < 0.01
print("check B passed")

**Your turn.** A quantity $T$ is uniform on $[2, 5]$ (in some unit). Find $P(T > 4)$ and $P(2.5 < T < 3)$.

*Hint:* the density is $1/3$; each probability is density × width.

In [ ]:
# TODO: write your answer here

## Problem 4 · What fraction of muons survives 3 μs?
The mean life of a muon at rest in vacuum is $\tau = 2.1970\ \mu\text{s}$ (PDG 2026). What is the probability that a muon survives longer than 3 μs?

**Predict.** About half, or about 0.4? Write your number. Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
rng = np.random.default_rng(4)
sim4 = (rng.exponential(TAU, 100_000) > 3).mean()
print(round(exp(-3 / TAU), 4), round(1 - 3 / TAU, 4), round(sim4, 4))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The second number is the linear first term $1 - t/\tau$: why is it wrong here?

In [ ]:
# check A (bounds): 0 <= S <= 1 and S(3 us) < S(1 us)
S3, S1 = exp(-3 / TAU), exp(-1 / TAU)
assert 0 <= S3 <= 1 and S3 < S1
print("check A passed")

In [ ]:
# check B (simulation): 100 000 decay times agree with the formula to 0.01
assert abs(sim4 - S3) < 0.01
print("check B passed")

**Your turn.** For the same muon, find the probability that the decay time lies between 3 μs and 5 μs.

*Hint:* a difference of two survival values, $e^{-3/\tau} - e^{-5/\tau}$.

In [ ]:
# TODO: write your answer here

## Problem 5 · A density from a histogram
A detector records 2000 muon decay times (simulated, $\tau = 2.1970\ \mu\text{s}$). In 0.5 μs bins the first bar holds 445 counts. What density does it estimate, and how does it compare with the decay law?

*Givens:* density estimate $= n/(N\,\Delta t)$ in 1/μs; true density $f(t) = e^{-t/\tau}/\tau$; the law averaged over 0–0.5 μs is $(1 - e^{-0.5/\tau})/0.5$.

**Predict.** Is the density 445, or $445/2000 = 0.22$? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
t = np.random.default_rng(5).exponential(TAU, 2000)          # us
n, edges = np.histogram(t, bins=np.arange(0, 12.25, 0.5))
dens = n / (2000 * 0.5)                                     # counts / (N dt), per us
truth0 = (1 - exp(-0.5 / TAU)) / 0.5                        # law averaged over the first bin
print(n[0], n[0] / (2000 * 0.5), n[0] / 2000)
print("decay law, bin average:", round(truth0, 4), "  f(0) =", round(1 / TAU, 4))

In [ ]:
import matplotlib.pyplot as plt
tt = np.linspace(0, 12, 400)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(edges[:-1], dens, width=0.5, align="edge", color="tab:blue", alpha=0.6, label="2000 decays: n / (N Δt)")
ax.plot(tt, np.exp(-tt / TAU) / TAU, "k", label="decay law f(t) = e^(-t/τ)/τ")
ax.set_xlabel("decay time t (μs)"); ax.set_ylabel("density (1/μs)")
ax.set_title("Problem 5 · counts become a density after dividing by N Δt")
ax.legend(); plt.tight_layout(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The third number is $n/N$ without the width: does it have units of 1/μs?

In [ ]:
# check A (area): the bar areas sum(density x dt) add to 0.996 over 0-12 us; the rest lies beyond 12 us
area = (dens * 0.5).sum()
print(round(area, 4))
assert 0.99 < area <= 1
print("check A passed")

In [ ]:
# check B (magnitude): sqrt(445) = 21.1 counts is 0.0211 per us; 0.445 vs 0.407 is a 1.8 sigma fluctuation
err = sqrt(n[0]) / (2000 * 0.5)
print(round(sqrt(n[0]), 1), round(err, 4), round((dens[0] - truth0) / err, 1))
assert abs(dens[0] - truth0) < 3 * err
print("check B passed")

**Your turn.** In the same histogram the bin 2.0–2.5 μs holds 163 counts. What density does it estimate, and what does the decay law give for that bin?

*Hint:* divide by $N\,\Delta t = 1000$ μs; the law's bin average is $(e^{-2/\tau} - e^{-2.5/\tau})/0.5$.

In [ ]:
# TODO: write your answer here

## Problem 6 · A voltmeter's error
A voltmeter reading scatters with Gaussian width $\sigma = 0.5$ V about the true voltage. What fraction of readings lies within 0.8 V of the true value?

**Predict.** About 68 %, as for one σ? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
from math import erf, sqrt
rng = np.random.default_rng(6)
err = rng.normal(0, 0.5, 1_000_000)
sim6 = (np.abs(err) < 0.8).mean()
print(round(erf(1.6 / sqrt(2)), 4), round(erf(1 / sqrt(2)), 3), round(sim6, 4))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The second number is the 1σ share. Why is it not the answer for 0.8 V?

In [ ]:
# check A (monotonicity): 68.3 % (1 sigma) < answer < 95.4 % (2 sigma)
ans = erf(1.6 / sqrt(2))
assert erf(1 / sqrt(2)) < ans < erf(2 / sqrt(2))
print("check A passed")

In [ ]:
# check B (simulation): one million draws agree with the formula to 0.001
assert abs(sim6 - ans) < 0.001
print("check B passed")

**Your turn.** For the same voltmeter, what fraction of readings lies within 1.0 V of the true value? Which band in σ is it?

*Hint:* 1.0 V is z = 2 for σ = 0.5 V; use the 2σ share.

In [ ]:
# TODO: write your answer here

## Problem 7 · A photomultiplier's dark counts
A photomultiplier in the dark gives 0, 1, 2, 3, 4 and "5 or more" counts in 1 ms windows; over 1000 windows these occur 301, 362, 216, 87, 26 and 8 times (illustrative). Estimate $P(X \ge 2)$ and $P(1 \le X \le 3)$.

*Givens:* the relative frequency $n_k/1000$ estimates $p(k)$; $F(k) = P(X \le k)$; for integers $P(a \le X \le b) = F(b) - F(a-1)$.

**Predict.** For $P(1 \le X \le 3)$, which two values of $F$ do you subtract? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
cnt = [301, 362, 216, 87, 26, 8]          # windows with 0..4, 5 or more counts
F = np.cumsum(cnt) / 1000
print(F.round(3))
print(round(F[3] - F[0], 3), round(F[3] - F[1], 3))
print("P(X >= 2) =", round(1 - F[1], 3))

In [ ]:
import matplotlib.pyplot as plt
k = np.arange(6)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(k, np.array(cnt) / 1000, color=["tab:gray", "tab:blue", "tab:blue", "tab:blue", "tab:gray", "tab:gray"],
       alpha=0.7, label="frequency p(k); bars 1-3 in blue")
ax.step(k, F, where="post", color="k", label="cumulative F(k)")
ax.set_xticks(k, ["0", "1", "2", "3", "4", "≥5"])
ax.set_xlabel("dark counts in a 1 ms window"); ax.set_ylabel("probability")
ax.set_title("Problem 7 · P(1 ≤ X ≤ 3) = F(3) − F(0)")
ax.legend(); plt.tight_layout(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The second number on the middle line is $F(3) - F(1)$: which windows does it drop?

In [ ]:
# check A (count): 362 + 216 + 87 = 665 windows of 1000
assert cnt[1] + cnt[2] + cnt[3] == 665 and abs((F[3] - F[0]) - 0.665) < 1e-12
print("check A passed")

In [ ]:
# check B (limits): the frequencies sum to 1 and F ends at 1.000
assert sum(cnt) == 1000 and abs(F[-1] - 1) < 1e-12
print("check B passed")

**Your turn.** From the same table, what fraction of windows had at most one count? What fraction had four or more?

*Hint:* "at most one" is $F(1)$; "four or more" is $1 - F(3)$.

In [ ]:
# TODO: write your answer here

## Problem 8 · A display rounds a Gaussian signal
A sensor's voltage is Gaussian with mean 10.3 mV and $\sigma = 0.3$ mV (illustrative). The digital display rounds to whole millivolts. What is the probability that it shows 10 mV? And 11 mV?

*Givens:* the display shows $k$ when $k - 0.5 \le x < k + 0.5$; $\Phi(z)$ is the standard Gaussian cumulative; $z = (x - \mu)/\sigma$.

**Predict.** Is it the density at 10 mV, or about 50 % since 10.3 lies between 10 and 11? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
Phi = lambda z: 0.5 * (1 + erf(z / sqrt(2)))
mu, s = 10.3, 0.3                          # mV
p = {k: Phi((k + 0.5 - mu) / s) - Phi((k - 0.5 - mu) / s) for k in range(9, 12)}
print({k: round(v, 4) for k, v in p.items()}, round(sum(p.values()), 4))
f = lambda x: np.exp(-0.5 * ((x - mu) / s) ** 2) / (s * sqrt(2 * np.pi))
print("trap, density f(10) =", round(f(10), 4), "per mV;  f(mu) =", round(f(mu), 4), "per mV")

In [ ]:
import matplotlib.pyplot as plt
xx = np.linspace(9, 12, 400)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(xx, f(xx), "k", label="voltage density (1/mV)")
ax.fill_between(xx, f(xx), where=(xx >= 9.5) & (xx < 10.5), color="tab:blue", alpha=0.4, label="display 10: area 0.744")
ax.fill_between(xx, f(xx), where=(xx >= 10.5) & (xx < 11.5), color="tab:orange", alpha=0.4, label="display 11: area 0.253")
ax.set_xlabel("sensor voltage (mV)"); ax.set_ylabel("density (1/mV)")
ax.set_title("Problem 8 · a reading's probability is an area, not a height")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The second line shows densities. Why can $f(\mu) = 1.33$ per mV exceed one?

In [ ]:
# check A (normalisation): readings 9, 10, 11 carry 1.0000; 8 and 12 are below 1e-4
p8, p12 = Phi((8.5 - mu) / s) - Phi((7.5 - mu) / s), Phi((12.5 - mu) / s) - Phi((11.5 - mu) / s)
assert abs(sum(p.values()) - 1) < 1e-4 and p8 < 1e-4 and p12 < 1e-4
print("check A passed")

In [ ]:
# check B (simulation): 100 000 simulated readings rounded (seed 8)
disp = np.round(np.random.default_rng(8).normal(mu, s, 100_000))
print(round((disp == 10).mean(), 4), round((disp == 11).mean(), 4))
assert abs((disp == 10).mean() - p[10]) < 0.01 and abs((disp == 11).mean() - p[11]) < 0.01
print("check B passed")

**Your turn.** The same display with a noisier sensor, $\sigma = 0.8$ mV (mean still 10.3 mV). What are $P(\text{display} = 10)$ and $P(\text{display} = 11)$?

*Hint:* same formula with $\sigma = 0.8$; the z-values are $(10.5 - 10.3)/0.8$ and $(9.5 - 10.3)/0.8$.

In [ ]:
# TODO: write your answer here

## Before you close the tab
- **Mistakes we met:** $6 \times 0.2 = 1.2$ (P1), the missing arrangement count (P2), one dead zone only (P3), the linear first term (P4), counts as a density (P5), the 68 % rule for any interval (P6), a dropped endpoint (P7), a density read as a probability (P8).
- **Each has a check:** probabilities between 0 and 1 and adding to one; a simulation with the same seed; units in every line.
- **Save your copy of Lab3.ipynb to Drive before you leave.** The Drive cell creates `PHY405_Labs/Week03`.

Next week: moments and moment generating functions (Week 4).